In [13]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Scan .log files, extract demographics from 'experiment_started' events,
and output both a CSV and a LaTeX table.
"""
from __future__ import annotations
import argparse
from pathlib import Path
import json
import sys
import pandas as pd

rename_map = {
    "user_id": "Participant ID",
    "session_id": "Session ID",
    "timestamp": "Start Time",
    "language": "Study Language",
    "system_order": "System Order (First/Second)",
    "age": "Age",
    "gender": "Gender",
    "occupation": "Occupation",
    "ai_tool_types": "AI Tool Types Used",
    "ai_frequency": "AI Usage Frequency",
    "ai_purposes": "AI Usage Purposes",
    "ai_proficiency": "Self-rated AI Proficiency (1–5)",
    "english_level": "English Proficiency",
    "programming_duration": "Programming Experience (years)",
    "programming_languages": "Programming Languages",
    "programming_contexts": "Programming Contexts",
    "programming_proficiency": "Self-rated Programming Proficiency (1-7)"
}

def parse_line_for_json(line: str):
    """
    Extract the first top-level JSON object from a log line.
    Strategy: find the first '{' and try json.loads.
    Return dict or None.
    """
    start = line.find('{')
    if start == -1:
        return None
    candidate = line[start:].strip()
    # Some logs have trailing text; try to find matching braces
    # We attempt progressively shorter endings.
    # But in most cases, json blob runs to EOL, so try straight loads first.
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        # try to trim trailing characters until we can decode
        # (simple brace-balance approach)
        depth = 0
        end_index = None
        for i, ch in enumerate(candidate):
            if ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    end_index = i + 1
                    break
        if end_index is not None:
            try:
                return json.loads(candidate[:end_index])
            except json.JSONDecodeError:
                return None
        return None

def flatten_demographics(demo: dict) -> dict:
    """
    Normalize demographics dictionary:
    - Convert list fields to comma-separated strings
    - Keep keys as-is; caller may rename for presentation
    """
    out = {}
    for k, v in demo.items():
        if isinstance(v, list):
            out[k] = ', '.join(map(str, v))
        else:
            out[k] = v
    return out

def collect_demographics(log_dir: Path, verbose: bool=False):
    rows = []
    for path in log_dir.rglob('*.log'):
        if verbose:
            print(f"Scanning {path}", file=sys.stderr)
        try:
            with path.open('r', encoding='utf-8', errors='replace') as f:
                for line in f:
                    if '"experiment_started"' not in line and '"demographic' not in line:
                        # quick filter
                        continue
                    data = parse_line_for_json(line)
                    if not isinstance(data, dict):
                        continue
                    # Expect structure like:
                    # {"timestamp": "...", "event":"experiment_started", "details": {...}, "platform":"...", "user_id":"..."}
                    event = data.get('event', '')
                    if event != 'experiment_started':
                        continue
                    details = data.get('details', {}) or {}
                    demo = details.get('demographic_data') or details.get('demographics') or {}
                    if not isinstance(demo, dict):
                        demo = {}
                    demo = flatten_demographics(demo)
                    row = {
                        'timestamp': data.get('timestamp'),
                        'user_id': data.get('user_id') or details.get('user_id'),
                        'session_id': details.get('session_id') or data.get('session_id'),
                        'language': details.get('language'),
                        'system_order': (details.get('demographic_data') or {}).get('system_order') \
                                         or (details.get('system_order')),
                        **demo
                    }
                    rows.append(row)
        except Exception as e:
            print(f"[WARN] Failed to process {path}: {e}", file=sys.stderr)
    return rows


log_dir = Path("./log")
rows = collect_demographics(log_dir)

df = pd.DataFrame(rows)

# Provide a sensible default column order if user didn't pass one
cols = ['user_id',
    'age', 'gender', 'occupation',
    'ai_tool_types', 
    'ai_frequency', 
    # 'ai_purposes', 
    # 'ai_proficiency',
    # 'english_level'
    'programming_duration', 
    # 'programming_languages', 
    # 'programming_contexts', 
    'programming_proficiency'
    ]
df = df[cols]

df = df.rename(columns=rename_map)

CATEGORIES = {
    "Text-based AI Experience": ["text", "chat", "language"],
    "Image Generation AI Experience": ["image", "vision", "graphics"]
}

# 임시 binary 컬럼 생성
for col, keywords in CATEGORIES.items():
    df[col] = df["AI Tool Types Used"].fillna("").apply(
        lambda x: any(k.lower() in x.lower() for k in keywords)
    ).map({True: "Y", False: "N"})

# 두 개를 합쳐서 하나의 열로
df["AI Exp*"] = df["Text-based AI Experience"] + "/" + df["Image Generation AI Experience"]

# 임시 컬럼 삭제 (원하면)
df = df.drop(columns=["Text-based AI Experience", "Image Generation AI Experience"])

    
df =df.drop(columns=["AI Tool Types Used"])
# 참가자별 gender 매핑 딕셔너리
gender_map = {
    "P1": "Female",
    "P2": "Male",
    "P3": "Male",
    "P4": "Male",
    "P5": "Male",
    "P6": "Male",
    "P7": "Male",
    "P8": "Female",
    "P9": "Female",
    "P10": "Female",
    "P11": "Male",
    "P12": "Female"
}

df["Gender"] = df["Participant ID"].map(gender_map)

occupation_map = {
    "학생": "Student",
    "학부생": "Undergraduate Student",
    "대학생": "Undergraduate Student",
    "대학원생": "Graduate Student",
    "연구원": "Researcher",
    "엔지니어": "Engineer",
    "디자이너": "Designer",
    "교사": "Teacher",
    "직장인": "Office Worker"
    # 필요하면 더 추가 가능
}

df["Occupation"] = df["Occupation"].map(lambda x: occupation_map.get(x, x))

cols = [
    "Participant ID",
    "Age",
    "Gender",
    "AI Exp*",
    # "Occupation",
    "AI Usage Frequency",
    "Programming Experience (years)",
    "Self-rated Programming Proficiency (1-7)",
]
df = df[cols]
# Participant ID에서 숫자만 추출해서 정렬 기준으로 사용
df["Participant Number"] = df["Participant ID"].str.extract(r'P(\d+)').astype(int)
df = df.sort_values(by="Participant Number").drop(columns=["Participant Number"])
# df = df.drop(columns=["Participant Number"])
# df에 Gender 컬럼 추가/덮어쓰기
df["Gender"] = df["Participant ID"].map(gender_map)

# Make a tidy LaTeX table
# NOTE: For Korean in LaTeX, compile with XeLaTeX or LuaLaTeX and a proper font.
# By default, escape=True to protect special chars. Set escape=False if you manage LaTeX safely.
print(df.to_latex(index=False, escape=True))
# Save
# with open("demographics.tex", "w", encoding="utf-8") as f:
#     f.write(latex_code)


\begin{tabular}{lllllll}
\toprule
Participant ID & Age & Gender & AI Exp* & AI Usage Frequency & Programming Experience (years) & Self-rated Programming Proficiency (1-7) \\
\midrule
P1 & 23 & Female & Y/Y & daily & 1-3 & 4 \\
P2 & 22 & Male & Y/Y & daily & >3 & 5 \\
P3 & 22 & Male & Y/Y & daily & 1-3 & 5 \\
P4 & 25 & Male & Y/N & weekly & never & 2 \\
P5 & 23 & Male & Y/Y & daily & never & 1 \\
P6 & 24 & Male & Y/Y & daily & >3 & 5 \\
P7 & 25 & Male & Y/Y & daily & <1 & 2 \\
P7 & 25 & Male & Y/Y & daily & <1 & 2 \\
P8 & 22 & Female & Y/N & weekly & never & 1 \\
P9 & 22 & Female & Y/Y & weekly & never & 2 \\
P10 & 26 & Female & Y/N & daily & <1 & 2 \\
P11 & 23 & Male & Y/N & monthly & never & 1 \\
P12 & 21 & Female & Y/Y & weekly & 1-3 & 4 \\
\bottomrule
\end{tabular}



In [14]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

mean_age = df["Age"].mean()
std_age = df["Age"].std()

print(f"Age: {mean_age:.2f} ± {std_age:.2f}")

df["Age"].min(), df["Age"].max()

Age: 23.31 ± 1.55


(21, 26)

In [15]:
col_="Self-rated Programming Proficiency (1-7)"

df[col_] = pd.to_numeric(df[col_], errors="coerce")

mean_age = df[col_].mean()
std_age = df[col_].std()

print(f"Age: {mean_age:.2f} ± {std_age:.2f}")

df[col_].min(), df[col_].max()

Age: 2.77 ± 1.59


(1, 5)

In [16]:

log_dir = Path("./log_study2")
rows = collect_demographics(log_dir)

df = pd.DataFrame(rows)

# Provide a sensible default column order if user didn't pass one
cols = ['user_id',
    'age', 'gender', 'occupation',
    'ai_tool_types', 
    'ai_frequency', 
    # 'ai_purposes', 
    # 'ai_proficiency',
    # 'english_level'
    'programming_duration', 'programming_languages', 'programming_contexts', 'programming_proficiency'
    ]
df = df[cols]

df = df.rename(columns=rename_map)

CATEGORIES = {
    "Text-based AI Experience": ["text", "chat", "language"],
    "Image Generation AI Experience": ["image", "vision", "graphics"]
}

# 임시 binary 컬럼 생성
for col, keywords in CATEGORIES.items():
    df[col] = df["AI Tool Types Used"].fillna("").apply(
        lambda x: any(k.lower() in x.lower() for k in keywords)
    ).map({True: "Y", False: "N"})

# 두 개를 합쳐서 하나의 열로
df["AI Exp*"] = df["Text-based AI Experience"] + "/" + df["Image Generation AI Experience"]

# 임시 컬럼 삭제 (원하면)
df = df.drop(columns=["Text-based AI Experience", "Image Generation AI Experience"])

    
df =df.drop(columns=["AI Tool Types Used"])
# 참가자별 gender 매핑 딕셔너리
gender_map = {
    "P1": "Female",
    "P2": "Male",
    "P3": "Male",
    "P4": "Female",
    "P5": "Female",
    "P6": "Female",
}

df["Gender"] = df["Participant ID"].map(gender_map)
import pandas as pd

# 참가자별 매핑 (P1~P6)
design_map = {
    "P1": {"Design Experience": "3–5 years", "Design Proficiency": 6},
    "P2": {"Design Experience": "≥5 years", "Design Proficiency": 5},
    "P3": {"Design Experience": "<1 year", "Design Proficiency": 2},
    "P4": {"Design Experience": "1–3 years", "Design Proficiency": 4},
    "P5": {"Design Experience": "None", "Design Proficiency": 1},
    "P6": {"Design Experience": "<1 year", "Design Proficiency": 4},
}

# DataFrame으로 변환
# df_design = pd.DataFrame.from_dict(design_map, orient="index").reset_index()
# df_design = df_design.rename(columns={"index": "Participant ID"})

# print(df_design)
df["Design Experience"] = df["Participant ID"].map(lambda x: design_map.get(x, {}).get("Design Experience", "N/A"))
df["Design Proficiency (1–7)"] = df["Participant ID"].map(lambda x: design_map.get(x, {}).get("Design Proficiency", "N/A"))

occupation_map = {
    "학생": "Student",
    "학부생": "Undergraduate Student",
    "대학생": "Undergraduate Student",
    "대학원생": "Graduate Student",
    "연구원": "Researcher",
    "엔지니어": "Engineer",
    "디자이너": "Designer",
    "교사": "Teacher",
    "직장인": "Office Worker"
    # 필요하면 더 추가 가능
}

df["Occupation"] = df["Occupation"].map(lambda x: occupation_map.get(x, x))

cols = [
    "Participant ID",
    "Age",
    "Gender",
    "AI Exp*",
    # "Occupation",
    "AI Usage Frequency",
    "Design Experience",
    "Design Proficiency (1–7)",
    "Programming Experience (years)",
    "Self-rated Programming Proficiency (1-7)"
]
df = df[cols]
# Participant ID에서 숫자만 추출해서 정렬 기준으로 사용
df["Participant Number"] = df["Participant ID"].str.extract(r'P(\d+)').astype(int)
df = df.sort_values(by="Participant Number").drop(columns=["Participant Number"])
# df = df.drop(columns=["Participant Number"])
# df에 Gender 컬럼 추가/덮어쓰기
df["Gender"] = df["Participant ID"].map(gender_map)

# Make a tidy LaTeX table
# NOTE: For Korean in LaTeX, compile with XeLaTeX or LuaLaTeX and a proper font.
# By default, escape=True to protect special chars. Set escape=False if you manage LaTeX safely.
print(df.to_latex(index=False, escape=True))
# Save
# with open("demographics.tex", "w", encoding="utf-8") as f:
#     f.write(latex_code)


\begin{tabular}{llllllrll}
\toprule
Participant ID & Age & Gender & AI Exp* & AI Usage Frequency & Design Experience & Design Proficiency (1–7) & Programming Experience (years) & Self-rated Programming Proficiency (1-7) \\
\midrule
P1 & 21 & Female & Y/Y & monthly & 3–5 years & 6 & 1-3 & 2 \\
P2 & 25 & Male & Y/Y & daily & ≥5 years & 5 & >3 & 7 \\
P3 & 22 & Male & Y/N & daily & <1 year & 2 & >3 & 6 \\
P4 & 22 & Female & Y/Y & daily & 1–3 years & 4 & <1 & 3 \\
P5 & 25 & Female & Y/Y & daily & None & 1 & >3 & 4 \\
P6 & 28 & Female & Y/Y & daily & <1 year & 4 & <1 & 2 \\
\bottomrule
\end{tabular}



In [17]:
df

,Participant ID,Age,Gender,AI Exp*,AI Usage Frequency,Design Experience,Design Proficiency (1–7),Programming Experience (years),Self-rated Programming Proficiency (1-7)
0,P1,21,Female,Y/Y,monthly,3–5 years,6,1-3,2
1,P2,25,Male,Y/Y,daily,≥5 years,5,>3,7
2,P3,22,Male,Y/N,daily,<1 year,2,>3,6
4,P4,22,Female,Y/Y,daily,1–3 years,4,<1,3
5,P5,25,Female,Y/Y,daily,None,1,>3,4
3,P6,28,Female,Y/Y,daily,<1 year,4,<1,2


In [18]:
col_="Self-rated Programming Proficiency (1-7)"

df[col_] = pd.to_numeric(df[col_], errors="coerce")

mean_age = df[col_].mean()
std_age = df[col_].std()

print(f"Age: {mean_age:.2f} ± {std_age:.2f}")

df[col_].min(), df[col_].max()

Age: 4.00 ± 2.10


(2, 7)

In [19]:
col_="Design Proficiency (1–7)"

df[col_] = pd.to_numeric(df[col_], errors="coerce")

mean_age = df[col_].mean()
std_age = df[col_].std()

print(f"Age: {mean_age:.2f} ± {std_age:.2f}")

df[col_].min(), df[col_].max()

Age: 3.67 ± 1.86


(1, 6)